# SynBioCrow seven-engine Colab review candidate

This notebook retains the six M1.16 adapters and runs ChemEnzyRetroPlanner locally with the M1.20.6 source pin. Read the acceptance report before comparing live and archived runs. Keep the SOURCE ZIP and execution evidence as separate files. The graph uses documented seed assumptions and candidate reactions.


In [ ]:
# M1.10.2: safe, idempotent Google Drive selection. Never modify /content/drive.
from pathlib import Path
import os
from google.colab import drive

def select_drive_root():
    for location in ('/content/gdrive', '/content/drive'):
        root = Path(location)
        if os.path.ismount(str(root)) and (root / 'MyDrive').is_dir():
            print('[DRIVE] Reusing mounted Drive:', root)
            return root
    # Prefer the already-established alternate mount point. Never mount over nonempty paths.
    for location in ('/content/gdrive', '/content/synbiocrow_gdrive'):
        root = Path(location)
        if root.is_symlink() or (root.exists() and not root.is_dir()):
            continue
        if root.is_dir() and any(root.iterdir()):
            # A MyDrive directory alone is not proof of an active mount.
            print('[DRIVE] Occupied but not mounted; leaving untouched:', root)
            continue
        root.mkdir(parents=True, exist_ok=True)
        drive.mount(str(root))
        if (root / 'MyDrive').is_dir():
            print('[DRIVE] Mounted:', root)
            return root
    raise RuntimeError('No safe Drive mount available; existing directories were preserved.')

DRIVE_ROOT = select_drive_root()
DRIVE_MYDRIVE = DRIVE_ROOT / "MyDrive"
assert DRIVE_MYDRIVE.is_dir(), f"Drive unavailable: {DRIVE_MYDRIVE}"
DRIVE_BASE = DRIVE_ROOT / 'MyDrive' / 'SynBioCrow' / '2.4'
print('[DRIVE] SynBioCrow root:', DRIVE_BASE)


In [ ]:
# Self-contained Drive initialization: works even if the preceding cell was skipped.
from pathlib import Path
import os
if 'DRIVE_MYDRIVE' not in globals() or not Path(DRIVE_MYDRIVE).is_dir():
    candidates = [Path('/content/gdrive/MyDrive'), Path('/content/drive/MyDrive'), Path('/content/synbiocrow_gdrive/MyDrive')]
    existing = [p for p in candidates if p.is_dir() and os.path.ismount(str(p.parent))]
    if not existing:
        # Mount only at an empty, dedicated location; never touch /content/drive.
        from google.colab import drive
        mountpoint = Path('/content/synbiocrow_gdrive')
        if mountpoint.exists() and any(mountpoint.iterdir()):
            raise RuntimeError('Drive mount not detected; /content/synbiocrow_gdrive is occupied. Run the Drive setup cell first.')
        mountpoint.mkdir(parents=True, exist_ok=True)
        drive.mount(str(mountpoint))
        existing = [mountpoint / 'MyDrive'] if (mountpoint / 'MyDrive').is_dir() else []
    if not existing:
        raise RuntimeError('Google Drive MyDrive unavailable; run the Drive setup cell first.')
    DRIVE_MYDRIVE = existing[0]
DRIVE_ROOT = Path(DRIVE_MYDRIVE).parent
DRIVE_BASE = Path(DRIVE_MYDRIVE) / 'SynBioCrow' / '2.4'
print('[M1.16 DRIVE]', DRIVE_MYDRIVE)

from pathlib import Path
import json, zipfile, hashlib, datetime, shutil, sys, subprocess, collections, csv, time
NAME='SynBioCrow_M1_20_7_SEVEN_ENGINE_REVIEW'
BASE=Path(DRIVE_MYDRIVE)/'SynBioCrow'/'2.4'/'M1_20_REVIEW'
BASE.mkdir(parents=True,exist_ok=True)
TARGETS=['3_hydroxypropionic_acid','butanal','hexanoic_acid']
TARGET_SMILES={'3_hydroxypropionic_acid':'O=C(O)CCO','butanal':'CCCC=O','hexanoic_acid':'CCCCCC(=O)O'}
ENGINES=['DORAnet','AiZynthFinder','ChemEnzyRetroPlanner','RetroTide','BioPKS','RetroBioCat2','RetroPath2']
TIMEOUT_PER_ENGINE=3600
# Set to True only when deliberately reanalyzing a verified prior M1.16 archive.
SOURCE_ZIP = Path('/content/SynBioCrow_M1_20_7_SOURCE.zip')
CHEMENZY_CACHE = Path(DRIVE_MYDRIVE) / 'SynBioCrow' / 'ChemEnzyRetroPlanner' / 'cache'
FETCH_MISSING_CHEMENZY_ASSETS = False
CHEMENZY_STOCK = "BioNav-stock"
CHEMENZY_ITERATIONS = 12
SEED_ARCHIVE = Path(DRIVE_MYDRIVE) / 'SynBioCrow' / '2.4' / 'M1_4' / 'SynBioCrow_V2_4_29_1_SEVEN_ENGINE_CONNECTIVITY_RECOVERY_M1_4_ALL_OUTPUTS.zip'
M116_EVIDENCE_ZIP = Path('/content/M1.16_ALL_OUTPUTS.zip')
print('[CONFIG]',BASE, 'targets',TARGETS)

## Restore the SOURCE ZIP

In [ ]:
from pathlib import PurePosixPath
import tempfile
SOURCE_ROOT = Path(tempfile.mkdtemp(prefix='synbiocrow_source_', dir='/content'))
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    for member in archive.namelist():
        relative = PurePosixPath(member)
        if relative.is_absolute() or '..' in relative.parts:
            raise ValueError(f'Unsafe source member {member}')
    archive.extractall(SOURCE_ROOT)
ROOT = SOURCE_ROOT / 'notebooks/seven_engine'
assert (ROOT / 'master_v2.py').is_file()
print('Source SHA256', hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest())


## Execute the native adapters

In [ ]:
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN = Path(tempfile.mkdtemp(prefix=RUN_ID + '_', dir='/content'))
OUT = RUN / 'outputs'
env = dict(os.environ, CHEMENZY_CACHE=str(CHEMENZY_CACHE),
           CHEMENZY_FETCH='1' if FETCH_MISSING_CHEMENZY_ASSETS else '0',
           CHEMENZY_STOCK=CHEMENZY_STOCK, CHEMENZY_ITERATIONS=str(CHEMENZY_ITERATIONS))
cmd = [sys.executable, '-u', str(ROOT / 'master_v2.py'), '--output', str(OUT),
       '--timeout', str(TIMEOUT_PER_ENGINE), '--targets', *TARGETS, '--engines', *ENGINES]
with (RUN / 'master_console.log').open('w') as log:
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, env=env) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
        master_exit = process.wait()
if not (OUT / 'summary.json').is_file():
    raise RuntimeError(f'Master did not produce its summary, exit {master_exit}')
shutil.copy2(RUN / 'master_console.log', OUT)
(OUT / 'master_exit.json').write_text(json.dumps({'exit_code': master_exit, 'run_id': RUN_ID}))


In [ ]:
print('Run evidence', OUT)


## Strict evidence accounting and reaction-direction checks

In [ ]:
try:
 from rdkit import Chem
except ImportError:
 subprocess.run([sys.executable,'-m','pip','-q','install','rdkit==2026.9.2'],check=True)
 from rdkit import Chem

def canon(s):
 m=Chem.MolFromSmiles(str(s))
 return Chem.MolToSmiles(m,canonical=True) if m and all(a.GetAtomicNum() for a in m.GetAtoms()) else None

def side(s):
 return [canon(x) for x in str(s or '').split('.') if x] if s else []
report=json.loads((OUT/'summary.json').read_text()) if (OUT/'summary.json').exists() else {}
records=json.loads((OUT/'fresh_native_reactions.json').read_text()) if (OUT/'fresh_native_reactions.json').exists() else []
if not isinstance(records,list):records=[]
EVID=OUT/'m116_acceptance';EVID.mkdir(exist_ok=True)
counts=collections.Counter(r.get('engine') for r in records if isinstance(r,dict))
engine_status=[]
for e in ENGINES:
 state=report.get('engine_states',{}).get(e,{})
 run=state.get('run',{}) if isinstance(state,dict) else {}
 process=run.get('status','NOT_REPORTED')
 engine_status.append({'engine':e,'process_status':process,'exit_code':run.get('exit_code'),'reaction_records':counts[e],
   'native_design_only':e in {'BioPKS','RetroTide'},'native_log':run.get('log'),
   'process_pass':process=='PROCESS_PASS' and run.get('exit_code')==0})
print('[ENGINE PROCESS STATUS]')
for x in engine_status:print(x)
usable=[];quarantined=[];invalid=[]
for i,r in enumerate(records):
 if not isinstance(r,dict):continue
 a,b=side(r.get('reactants')),side(r.get('products'))
 if not a or not b or any(x is None for x in a+b):
  invalid.append({'index':i,'reason':'invalid SMILES, unresolved placeholder or missing side'});continue
 row={'index':i,'engine':r.get('engine'),'reactants':a,'products':b,'native_file':r.get('native_file'),'reaction_smiles':r.get('reaction_smiles'),
     'native_sha256':r.get('native_sha256'), 'native_pointer':r.get('native_pointer'), 'direction':r.get('direction'),
     'native_direction':r.get('native_direction'), 'native_reaction':r.get('native_reaction')}
 if row['engine']=='RetroPath2':
  quarantined.append({**row,'reason':'target synthesis direction not independently verified'});continue
 if row['engine']=='BioPKS':
  quarantined.append({**row,'reason':'design/release adapter is not a validated balanced precursor-producing reaction'});continue
 usable.append(row)
print('[RECORDS]',len(records),'usable candidate directions',len(usable),'quarantined',len(quarantined),'invalid',len(invalid))

## Recover the original seed set; require exact source provenance

In [ ]:
seed_zip = SEED_ARCHIVE
seed_hash = 'd25c756fc8980bb8aff64494c006e85c59c1d6a64c2a4140ff953d23ea85346c'
if seed_zip.is_file():
    if hashlib.sha256(seed_zip.read_bytes()).hexdigest() != seed_hash:
        raise ValueError('M1.4 seed archive SHA256 mismatch')
    with zipfile.ZipFile(seed_zip) as archive:
        seed_raw = json.loads(archive.read('results/summary.json'))['assumed_seeds']
    seed_record = {'archive': str(seed_zip), 'archive_sha256': seed_hash}
else:
    seed_zip = M116_EVIDENCE_ZIP
    evidence_digest = 'adb0a3ea2cd381caffcad1a3b3341e6450548bb2907d521adbdeff8d9a100078'
    if hashlib.sha256(seed_zip.read_bytes()).hexdigest() != evidence_digest:
        raise ValueError('M1.16 evidence archive SHA256 mismatch')
    with zipfile.ZipFile(seed_zip) as archive:
        member = 'outputs/m116_acceptance/seed_provenance.json'
        description = json.loads(archive.read(member))
    assert description['archive_sha256'] == seed_hash
    seed_raw = description['seeds']
    seed_record = {'archive': str(seed_zip), 'archive_sha256': evidence_digest,
                   'member': member, 'reported_m1_4_archive_sha256': seed_hash}
seeds = {canon(smiles) for smiles in seed_raw}
if None in seeds or len(seeds) != 27:
    raise ValueError('Seed archive did not provide 27 valid distinct seeds')


## All-substrate graph search, reconstruct paths, identify real hybrid witnesses

In [ ]:
available=set(seeds);activated=[];remaining=list(usable);produced_by={};levels={s:0 for s in seeds}
for depth in range(1,12):
 ready=[r for r in remaining if set(r['reactants']).issubset(available)]
 if not ready:break
 new=set()
 for r in ready:
  activated.append(r)
  for p in r['products']:
   if p not in available and p not in new:
    produced_by[p]=r;levels[p]=depth;new.add(p)
 available.update(new)
 remaining=[r for r in remaining if r not in ready]
 print('[DEPTH]',depth,'activated',len(ready),'new molecules',len(new))
 if not new:break

def witness(target):
 target=canon(target)
 if target not in available:return None
 selected={};visiting=set()
 def visit(mol):
  if mol in seeds:return True
  if mol in visiting:return False
  r=produced_by.get(mol)
  if r is None:return False
  visiting.add(mol)
  ok=all(visit(s) for s in r['reactants'])
  visiting.remove(mol)
  if ok:selected[r['index']]=r
  return ok
 if not visit(target):return None
 ordered=sorted(selected.values(),key=lambda r:(max(levels.get(x,0) for x in r['reactants']),r['index']))
 return {'steps':ordered,'engines':sorted({r['engine'] for r in ordered}), 'multi_engine':len({r['engine'] for r in ordered})>1}
results={}
for name,smiles in TARGET_SMILES.items():
 w=witness(smiles)
 results[name]={'smiles':smiles,'reachable':canon(smiles) in available,'witness':w}
 print('[TARGET]',name,'reachable',results[name]['reachable'],'engines',w['engines'] if w else [],'steps',len(w['steps']) if w else 0)
# Also search every reachable product for multi-engine witnesses, without claiming they reach target.
junction_witnesses=[]
for mol in sorted(available-seeds):
 w=witness(mol)
 if w and w['multi_engine']:
  junction_witnesses.append({'product':mol,'witness':w})
print('[MULTI-ENGINE REACHABLE PRODUCTS]',len(junction_witnesses))
print('[MULTI-ENGINE TARGET PATHWAYS]',sum(bool(r['witness'] and r['witness']['multi_engine']) for r in results.values()))

## Explicit seven-engine acceptance gates and inspectable evidence

In [ ]:
target_matrix=[]
for e in ENGINES:
 erows=[r for r in usable if r['engine']==e]
 for target,smiles in TARGET_SMILES.items():
  c=canon(smiles)
  direct=sum(c in r['products'] for r in erows)
  witness_engine=bool(results[target]['witness'] and e in results[target]['witness']['engines'])
  target_matrix.append({'engine':e,'target':target,'direction_supported_product_records':direct,
    'engine_in_reconstructed_target_path':witness_engine,
    'native_pathway_proven_for_this_target':witness_engine and e not in {'RetroTide','BioPKS','RetroPath2'}})
acceptance={
 'all_seven_processes_passed':all(x['process_pass'] for x in engine_status),
 'all_reaction_engines_have_direction_admitted_candidates':all(any(r['engine']==e for r in usable) for e in ENGINES if e not in {'BioPKS','RetroTide'}),
 'pks_design_evidence':{e:report.get('engine_states',{}).get(e,{}).get('specialist_evidence',{}) for e in ('BioPKS','RetroTide')},
 'all_reaction_engines_have_reconstructed_target_pathways':all(any(x['engine']==e and x['engine_in_reconstructed_target_path'] for x in target_matrix) for e in ENGINES if e not in {'BioPKS','RetroTide'}),
 'complete_multi_engine_target_pathway_demonstrated':any(r['witness'] and r['witness']['multi_engine'] for r in results.values()),
 'biochemical_certification':False,
 'note':'Graph witnesses rely on assumed starting materials and recorded candidate reaction direction. No claim of balanced reactions or experimentally verified feasibility.'}
for filename,obj in [('engine_status.json',engine_status),('target_engine_matrix.json',target_matrix),('target_witnesses.json',results),('multi_engine_junction_witnesses.json',junction_witnesses[:1000]),('acceptance_gates.json',acceptance),('quarantined_reactions.json',quarantined),('invalid_reactions.json',invalid),('seed_provenance.json',{'count':len(seeds),**seed_record,'seeds':sorted(seeds)}),('usable_reactions.json',usable)]:
 (EVID/filename).write_text(json.dumps(obj,indent=2))
with (EVID/'target_engine_matrix.csv').open('w',newline='') as f:
 w=csv.DictWriter(f,fieldnames=list(target_matrix[0]));w.writeheader();w.writerows(target_matrix)
print('[ACCEPTANCE GATES]',json.dumps(acceptance,indent=2))

## Persist full native logs and acceptance reports; download to laptop

In [ ]:
ARCHIVE=BASE/(NAME+'_'+RUN_ID+'_ALL_OUTPUTS.zip')
# Full native run evidence, not a recycled historic summary; do not package nested prior archives.
with zipfile.ZipFile(ARCHIVE,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
 for p in sorted(OUT.rglob('*')):
  if p.is_file():z.write(p,'outputs/'+str(p.relative_to(OUT)))
with zipfile.ZipFile(ARCHIVE) as z:
 assert z.testzip() is None
 assert 'outputs/summary.json' in z.namelist()
sha=hashlib.sha256(ARCHIVE.read_bytes()).hexdigest()
print('[SAVED]',ARCHIVE,'bytes',ARCHIVE.stat().st_size,'SHA256',sha)
print('[IMPORTANT] Native process success is distinct from seven-engine pathway acceptance.')
try:
 from google.colab import files
 files.download(str(ARCHIVE))
except Exception as exc:
 print('[DOWNLOAD] Browser download unavailable:',repr(exc))
 print('[RECOVERY] Archive is safely in Drive:',ARCHIVE)